<div dir="rtl" align="right">

# مصروفي · درّب نموذجك بنفسك

بنعلّم نموذج **Qwen3‑1.7B** يحوّل كلامك العادي إلى بيانات مرتبة:

> «تغديت ب٤٠ أمس وعبيت بنزين ب٨٠»  →  `{"expenses": [{"item": "غدا", "amount": 40, ...}, {"item": "بنزين", ...}]}`

| الخطوة | الوقت تقريباً |
|---|---|
| ٠. تشغيل GPU وتثبيت الأدوات | ٤ دقائق |
| ١. تجهيز البيانات | دقيقة |
| ٢. اختبار النموذج **قبل** التدريب | دقيقتين |
| ٣. التدريب | ٤ دقائق |
| ٤. اختبار النموذج **بعد** التدريب | دقيقتين |
| ٥. التصدير إلى Ollama | ١٠ دقائق |

**قبل لا تبدأ:** من القائمة **Runtime → Change runtime type** اختر **T4 GPU** واضغط Save.
بعدها شغّل الخلايا بالترتيب، كل خلية بزر ▶ أو **Shift + Enter**.

لو طلع لك خطأ `Torch not compiled with CUDA enabled` أو `no GPU found`، معناه إنك على CPU: غيّر الـ Runtime لـ GPU وشغّل من البداية.

</div>

<div dir="rtl" align="right">

## ٠. تأكد إن الـ GPU شغّال وثبّت الأدوات

</div>

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("ما فيه GPU ❌  من القائمة: Runtime ← Change runtime type ← اختر T4 GPU ← Save، وبعدها شغّل الدفتر من البداية")
print("GPU شغّال ✅", torch.cuda.get_device_name(0))

In [ ]:
%%capture
# نفس خلية التثبيت في دفاتر Unsloth الرسمية لكولاب (تاخذ ٣ إلى ٤ دقائق)
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

<div dir="rtl" align="right">

## ١. البيانات

الملف تحت يولّد ٧٠٠ رسالة تدريب و١١٦ رسالة اختبار بلهجتنا: أرقام عربية وإنجليزية، مبالغ بالكلام مثل «خمسة وعشرين»،
رسالتين في سطر، ورسائل **مو** مصروف (سؤال، خطة، راتب) لازم يرجع لها قائمة فاضية.

رسائل الاختبار فيها أغراض ما شافها النموذج وقت التدريب، على شان نقيس إنه **فهم** مو **حفظ**.

</div>

In [ ]:
%%writefile make_data.py
# -*- coding: utf-8 -*-
"""Generate Gulf-Arabic expense messages with their JSON labels.

    python make_data.py            -> data/train.jsonl, data/test.jsonl

Every line is {"text": ..., "label": {"expenses": [...]}}.
An expense is {"item", "amount", "category", "when"}; a message that records
no money spent (a question, a plan, income, a greeting) has "expenses": [].
"""
import json
import os
import random

CATEGORIES = ["مطاعم", "قهوة", "بقالة", "مواصلات", "فواتير", "تسوق", "صحة", "ترفيه", "أخرى"]

SYSTEM = (
    "أنت تسجّل المصاريف. حوّل رسالة المستخدم إلى JSON فقط بهذا الشكل:\n"
    '{"expenses": [{"item": "...", "amount": رقم, "category": "...", "when": "اليوم|أمس|قبل أمس"}]}\n'
    "التصنيفات المسموحة: " + "، ".join(CATEGORIES) + ".\n"
    "إذا لم يذكر الوقت فهو اليوم. إذا الرسالة ليست مصروفاً دُفع فعلاً "
    '(سؤال، خطة، دخل، تحية) أعد {"expenses": []}.'
)

# item as the user writes it -> category
ITEMS = {
    "قهوة": ["قهوة", "لاتيه", "كابتشينو", "سبانش لاتيه", "شاي كرك", "موكا", "قهوة مختصة", "فلات وايت", "آيس لاتيه", "دلة قهوة"],
    "مطاعم": ["غدا", "عشا", "فطور", "برجر", "شاورما", "مندي", "كبسة", "بيتزا", "سوشي", "مطبق", "فلافل", "بروستد"],
    "بقالة": ["بقالة", "خضار", "حليب", "خبز", "مقاضي البيت", "فواكه", "رز", "تمر", "موية"],
    "مواصلات": ["بنزين", "أوبر", "كريم", "تاكسي", "موقف", "تذكرة مترو", "غسيل السيارة", "تغيير زيت"],
    "فواتير": ["فاتورة الكهرباء", "فاتورة الجوال", "فاتورة النت", "فاتورة الموية", "شحن رصيد", "إيجار"],
    "تسوق": ["ثوب", "شماغ", "جزمة", "عطر", "ملابس", "ساعة", "سماعة", "شنطة", "نظارة"],
    "صحة": ["صيدلية", "دواء", "كشفية", "فيتامينات", "تحليل دم", "بنادول", "نادي رياضي"],
    "ترفيه": ["سينما", "بلايستيشن", "اشتراك نتفلكس", "تذاكر مباراة", "بولينج", "اشتراك سبوتيفاي", "ملاهي"],
    "أخرى": ["هدية", "تبرع", "حلاق", "كي الملابس", "طوابع"],
}

# paid for, not bought: "اشتريت تاكسي" reads wrong
SERVICES = {"أوبر", "كريم", "تاكسي", "موقف", "غسيل السيارة", "تغيير زيت", "فاتورة الكهرباء", "فاتورة الجوال",
            "فاتورة النت", "فاتورة الموية", "إيجار", "كشفية", "تحليل دم", "نادي رياضي", "سينما", "بولينج",
            "ملاهي", "تبرع", "حلاق", "كي الملابس", "غدا", "عشا", "فطور", "اشتراك نتفلكس", "اشتراك سبوتيفاي", "بنزين"}

# a few items held out of training so the test set checks generalisation
HELD_OUT = {"فلات وايت", "مطبق", "تمر", "تغيير زيت", "إيجار", "نظارة", "نادي رياضي", "ملاهي", "طوابع"}

WORDS = {10: "عشرة", 15: "خمسطعش", 20: "عشرين", 25: "خمسة وعشرين", 30: "ثلاثين", 40: "أربعين",
         50: "خمسين", 60: "ستين", 70: "سبعين", 100: "مية", 150: "مية وخمسين", 200: "ميتين",
         300: "ثلاث مية", 500: "خمس مية", 1000: "ألف"}

TYPICAL = {"قهوة": (9, 35), "مطاعم": (15, 220), "بقالة": (5, 450), "مواصلات": (10, 180),
           "فواتير": (40, 900), "تسوق": (50, 1500), "صحة": (15, 400), "ترفيه": (20, 300), "أخرى": (10, 300)}

WHEN = [("", "اليوم"), ("اليوم", "اليوم"), ("الحين", "اليوم"), ("أمس", "أمس"), ("امس", "أمس"),
        ("البارحة", "أمس"), ("قبل أمس", "قبل أمس"), ("أول أمس", "قبل أمس")]

CURRENCY = ["ريال", "ريال", "ريال", "", "ر.س", "ريال سعودي", "SAR"]

ITEM_RANGE = {"حليب": (4, 25), "خبز": (2, 15), "موية": (3, 30), "رز": (20, 90), "تمر": (15, 120),
              "فواكه": (10, 80), "خضار": (10, 90), "شاي كرك": (2, 8), "بنادول": (5, 20), "موقف": (3, 30),
              "تذكرة مترو": (4, 20), "شحن رصيد": (20, 200), "إيجار": (1500, 4000), "حلاق": (20, 80),
              "كي الملابس": (10, 60), "طوابع": (5, 30), "فلافل": (5, 25), "شاورما": (8, 40), "دواء": (10, 150),
              "بروستد": (20, 60), "مطبق": (10, 30), "فطور": (10, 60), "دلة قهوة": (25, 60), "عطر": (90, 900),
              "ساعة": (150, 1500), "ثوب": (90, 450), "شماغ": (60, 350), "اشتراك نتفلكس": (25, 65),
              "اشتراك سبوتيفاي": (20, 30), "تبرع": (10, 500), "بنزين": (30, 150), "أوبر": (12, 90), "كريم": (12, 90)}

INDIC = str.maketrans("0123456789.", "٠١٢٣٤٥٦٧٨٩٫")


def pick_amount(rng, cat, item=None):
    lo, hi = ITEM_RANGE.get(item, TYPICAL[cat])
    if rng.random() < 0.22:
        options = [v for v in WORDS if lo <= v <= hi] or [lo]
        value = rng.choice(options)
        return value, WORDS.get(value, str(value))
    if rng.random() < 0.15 and hi < 100:
        value = round(rng.uniform(lo, hi) * 2) / 2      # e.g. 12.5
        value = int(value) if value.is_integer() else value
    else:
        value = rng.randint(lo, hi)
    shown = f"{value:g}"
    if rng.random() < 0.3:
        shown = shown.translate(INDIC)
    return value, shown


def money(rng, shown):
    cur = rng.choice(CURRENCY)
    return f"{shown} {cur}".strip()


# templates take (item, amount text, when text); the item may be rewritten by the template
TEMPLATES = [
    "دفعت {m} {i} {w}",
    "{i} ب{m} {w}",
    "اشتريت {i} ب{m} {w}",
    "{w} اشتريت {i} ب{m}",
    "صرفت {m} على {i} {w}",
    "طلع علي {i} {m} {w}",
    "{i} {m}",
    "كلفني {i} {m} {w}",
    "{w} دفعت حق {i} {m}",
    "سجل {i} {m} {w}",
    "حاسبت {i} ب{m} {w}",
    "راحت {m} على {i} {w}",
]

VERB_ITEMS = {  # natural verbs that imply an item
    "مطاعم": [("تغديت ب{m} {w}", "غدا"), ("تعشيت ب{m} {w}", "عشا"), ("فطرت ب{m} {w}", "فطور")],
    "مواصلات": [("عبيت بنزين ب{m} {w}", "بنزين"), ("ركبت أوبر ب{m} {w}", "أوبر")],
    "فواتير": [("سددت فاتورة الكهرباء {m} {w}", "فاتورة الكهرباء"), ("شحنت رصيد ب{m} {w}", "شحن رصيد")],
    "قهوة": [("تقهويت ب{m} {w}", "قهوة")],
}

NOT_EXPENSE = [
    "كم صرفت هالشهر؟", "وش أكثر شي أصرف عليه؟", "السلام عليكم", "شكراً", "هلا",
    "ذكرني أدفع فاتورة الكهرباء بكرة", "أبي أشتري جوال ب{m}", "ناوي أشتري {i} ب{m} الأسبوع الجاي",
    "استلمت راتبي {m}", "جاني تحويل {m} من أخوي", "رجعوا لي {m} من {i}", "كم سعر {i}؟",
    "لو اشتريت {i} ب{m} يكون غالي؟", "امسح آخر مصروف", "ابي ملخص الأسبوع", "وش رصيدي؟",
    "بكرة بدفع {m} {i}", "خويي دفع عني {i}", "عطني نصيحة أوفر", "صديقي صرف {m} على {i}",
]


def clean(s):
    return " ".join(s.split())


def one_expense(rng, allow):
    cat = rng.choice(CATEGORIES)
    if cat in VERB_ITEMS and rng.random() < 0.3:
        tpl, item = rng.choice(VERB_ITEMS[cat])
    else:
        tpl, item = rng.choice(TEMPLATES), rng.choice(ITEMS[cat])
    if "اشتريت" in tpl and item in SERVICES:
        tpl = "دفعت {m} {i} {w}"
    if not allow(item):
        return None
    value, shown = pick_amount(rng, cat, item)
    w_text, w_val = rng.choice(WHEN) if "{w}" in tpl else ("", "اليوم")
    text = tpl.format(i=item, m=money(rng, shown), w=w_text)
    return clean(text), {"item": item, "amount": value, "category": cat, "when": w_val}


def sample(rng, allow):
    r = rng.random()
    if r < 0.14:                              # not an expense
        tpl = rng.choice(NOT_EXPENSE)
        cat = rng.choice(CATEGORIES)
        _, shown = pick_amount(rng, cat)
        text = tpl.format(m=money(rng, shown), i=rng.choice(ITEMS[cat]))
        return clean(text), {"expenses": []}
    if r < 0.30:                              # two expenses in one message
        a, b = one_expense(rng, allow), one_expense(rng, allow)
        if not a or not b:
            return None
        joiner = rng.choice([" و", " و ", "، و", " وبعدين "])
        return clean(a[0] + joiner + b[0]), {"expenses": [a[1], b[1]]}
    e = one_expense(rng, allow)
    return (e[0], {"expenses": [e[1]]}) if e else None


def build(n, seed, allow):
    rng, rows, seen = random.Random(seed), [], set()
    while len(rows) < n:
        s = sample(rng, allow)
        if s and s[0] not in seen:
            seen.add(s[0])
            rows.append({"text": s[0], "label": s[1]})
    return rows


def main(out="data", n_train=700, n_test=120):
    os.makedirs(out, exist_ok=True)
    train = build(n_train, 7, lambda item: item not in HELD_OUT)
    test = build(n_test, 99, lambda item: True)
    train_texts = {r["text"] for r in train}
    test = [r for r in test if r["text"] not in train_texts]
    for name, rows in (("train", train), ("test", test)):
        with open(os.path.join(out, f"{name}.jsonl"), "w", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"train: {len(train)}  test: {len(test)}  -> {out}/")


if __name__ == "__main__":
    main()

In [ ]:
!python make_data.py
import json, random
train = [json.loads(l) for l in open("data/train.jsonl", encoding="utf-8")]
test = [json.loads(l) for l in open("data/test.jsonl", encoding="utf-8")]
for r in random.sample(train, 5):
    print(r["text"], "\n   →", json.dumps(r["label"], ensure_ascii=False), "\n")

<div dir="rtl" align="right">

### أضف أمثلتك (اختياري)

اكتب جمل بأسلوبك أنت. النموذج يتعلم من **الأمثلة**، فكل مثال تضيفه يقرّبه من طريقتك في الكلام.

</div>

In [ ]:
MY_EXAMPLES = [
    ("دفعت ١٢ ريال كرك وسمبوسة", [{"item": "كرك وسمبوسة", "amount": 12, "category": "قهوة", "when": "اليوم"}]),
    ("أمس خلصت فاتورة الجوال ٢٣٠", [{"item": "فاتورة الجوال", "amount": 230, "category": "فواتير", "when": "أمس"}]),
    ("وش صرفت على القهوة هالشهر؟", []),
]
for text, expenses in MY_EXAMPLES:
    for _ in range(3):  # نكرر أمثلتك ٣ مرات على شان يكون لها وزن
        train.append({"text": text, "label": {"expenses": expenses}})
print("أمثلة التدريب:", len(train))

<div dir="rtl" align="right">

## ٢. حمّل النموذج وجهّز LoRA

نحمّل النموذج بدقة ٤‑بت على شان يدخل في ذاكرة T4. ما راح نعدّل ١.٧ مليار رقم،
بنضيف عليه طبقات **LoRA** صغيرة (أقل من ٢٪ من حجمه) وهي اللي تتدرّب.

طبقات LoRA تبدأ بأصفار، يعني النموذج الحين يتصرف **نفس الأصلي بالضبط**، وهذا اللي بنقيسه قبل التدريب.

</div>

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen3-1.7B"   # أسرع: "unsloth/Qwen3-0.6B"   أدق: "unsloth/Qwen3-4B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 1024,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
model.print_trainable_parameters()

<div dir="rtl" align="right">

## ٣. اختبر النموذج قبل التدريب

</div>

In [ ]:
import re
from make_data import SYSTEM

def ask(text, max_new_tokens = 200):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = True,
                                           enable_thinking = False)
    inputs = tokenizer(prompt, return_tensors = "pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens = max_new_tokens, do_sample = False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)

def parse(reply):
    reply = re.sub(r"<think>.*?</think>", "", reply, flags = re.S)
    start, end = reply.find("{"), reply.rfind("}")
    try:
        data = json.loads(reply[start:end + 1])
        return data if isinstance(data.get("expenses"), list) else None
    except Exception:
        return None

def same(pred, gold):
    p, g = pred.get("expenses", []), gold["expenses"]
    if len(p) != len(g):
        return False
    for a, b in zip(p, g):
        try:
            if abs(float(a["amount"]) - b["amount"]) > 0.01:
                return False
        except Exception:
            return False
        if a.get("category") != b["category"] or a.get("when", "اليوم") != b["when"]:
            return False
    return True

def evaluate(rows, show = 4):
    FastLanguageModel.for_inference(model)
    ok = right = 0
    misses = []
    for r in rows:
        reply = ask(r["text"])
        data = parse(reply)
        ok += data is not None
        if data is not None and same(data, r["label"]):
            right += 1
        else:
            misses.append((r["text"], reply.strip()[:160]))
    for text, reply in misses[:show]:
        print("✗", text, "\n   قال:", reply, "\n")
    n = len(rows)
    print(f"JSON صالح: {ok / n:.0%}   إجابات صحيحة: {right / n:.0%}")
    return {"json": ok / n, "correct": right / n}

EVAL = test[:40]
before = evaluate(EVAL)

<div dir="rtl" align="right">

## ٤. التدريب

كل مثال يتحول إلى محادثة: تعليمات النظام، ثم رسالتك، ثم الجواب الصحيح.
النموذج يتعلّم من **الجواب فقط** (`train_on_responses_only`)، ما نبيه يحفظ التعليمات.

راقب رقم **loss**: كل ما نزل، كان النموذج أقرب للإجابات الصحيحة.

</div>

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def to_text(r):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": r["text"]},
        {"role": "assistant", "content": json.dumps(r["label"], ensure_ascii = False)},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize = False)}

dataset = Dataset.from_list([to_text(r) for r in train])
print(dataset[0]["text"])

In [ ]:
FastLanguageModel.for_training(model)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        warmup_steps = 5,
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        logging_steps = 10,
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)
stats = trainer.train()
print(f"خلص التدريب في {stats.metrics['train_runtime'] / 60:.1f} دقيقة")

<div dir="rtl" align="right">

## ٥. اختبر بعد التدريب وقارن

في تجربتنا ارتفعت الإجابات الصحيحة من حوالي **١٨٪** إلى حوالي **٧٥٪**. أرقامك ممكن تختلف شوي.

</div>

In [ ]:
after = evaluate(EVAL)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize = (6, 3.2))
labels = ["valid JSON", "correct"]
x = range(len(labels))
ax.bar([i - 0.2 for i in x], [before["json"], before["correct"]], 0.4, label = "before", color = "#9AA0A6")
ax.bar([i + 0.2 for i in x], [after["json"], after["correct"]], 0.4, label = "after", color = "#F9AB00")
ax.set_xticks(list(x), labels); ax.set_ylim(0, 1); ax.legend(); ax.set_title("Qwen3-1.7B on 40 unseen messages")
plt.show()

<div dir="rtl" align="right">

**اقرأ الأخطاء الباقية:** أغلبها أكلات ما شافها النموذج في التدريب (مثل «بروستد» أو «سوشي») فيصنّفها «تسوق» بدل «مطاعم».
هذا حد البيانات مو حد النموذج: أضف أمثلة فيها هالأكلات في خلية «أضف أمثلتك» وأعد التدريب.

</div>

In [ ]:
# جرّب بنفسك: اكتب أي جملة، واكتب «خلاص» للخروج
while True:
    text = input("رسالتك: ").strip()
    if text in ("", "خلاص"):
        break
    print(parse(ask(text)) or "ما رجّع JSON")

<div dir="rtl" align="right">

## ٦. احفظ النموذج وصدّره إلى Ollama

١. نحفظ طبقات LoRA (حوالي ٧٠ ميقا) لو حبيت تكمل تدريب بعدين.
٢. ندمجها مع النموذج ونحوّله إلى ملف **GGUF** مضغوط (حوالي ١.١ قيقا). هذي الخطوة تاخذ ٣ إلى ١٠ دقائق، خلها تشتغل لين يطلع لك حجم الملف.

</div>

In [ ]:
model.save_pretrained("masrufi_lora")
tokenizer.save_pretrained("masrufi_lora")

model.save_pretrained_gguf("masrufi", tokenizer, quantization_method = "q4_k_m")

import glob, shutil
ggufs = [p for p in glob.glob("**/*.gguf", recursive = True) if "q4_k_m" in p.lower()]
shutil.copy(ggufs[0], "masrufi.Q4_K_M.gguf")
open("Modelfile", "w", encoding = "utf-8").write('# ضع ملف GGUF اللي نزلته من كولاب في نفس المجلد، وعدّل الاسم تحت لو اختلف\nFROM ./masrufi.Q4_K_M.gguf\n\n# نفس قالب المحادثة اللي تدرّب عليه النموذج (ChatML مع خانة تفكير فاضية)\nTEMPLATE """{{- range .Messages }}<|im_start|>{{ .Role }}\n{{ .Content }}<|im_end|>\n{{ end }}<|im_start|>assistant\n<think>\n\n</think>\n\n"""\n\nSYSTEM """أنت تسجّل المصاريف. حوّل رسالة المستخدم إلى JSON فقط بهذا الشكل:\n{"expenses": [{"item": "...", "amount": رقم, "category": "...", "when": "اليوم|أمس|قبل أمس"}]}\nالتصنيفات المسموحة: مطاعم، قهوة، بقالة، مواصلات، فواتير، تسوق، صحة، ترفيه، أخرى.\nإذا لم يذكر الوقت فهو اليوم. إذا الرسالة ليست مصروفاً دُفع فعلاً (سؤال، خطة، دخل، تحية) أعد {"expenses": []}."""\n\nPARAMETER stop "<|im_end|>"\nPARAMETER stop "<|im_start|>"\nPARAMETER temperature 0\nPARAMETER num_ctx 2048\n')
!ls -lh masrufi.Q4_K_M.gguf Modelfile

<div dir="rtl" align="right">

## ٧. نزّل النموذج على جهازك

⚠️ كولاب يمسح كل الملفات إذا انقطعت الجلسة، فانسخ النموذج قبل لا تقفل الصفحة.

**الطريقة الأسهل:** انسخه إلى Google Drive وحمّله من هناك (أسرع من التحميل المباشر لملف كبير).
بتطلع لك نافذة تطلب صلاحية Drive: اضغط **Connect to Google Drive** واختر حسابك.

</div>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/masrufi"
!cp masrufi.Q4_K_M.gguf Modelfile "/content/drive/MyDrive/masrufi/"
!cp -r masrufi_lora "/content/drive/MyDrive/masrufi/"
print("تم. افتح Google Drive ← مجلد masrufi ونزّل الملفين: masrufi.Q4_K_M.gguf و Modelfile")

<div dir="rtl" align="right">

**بديل:** ارفعه على Hugging Face وشغّله مباشرة بأمر واحد.
أضف مفتاحك من 🔑 **Secrets** في الشريط الجانبي باسم `HF_TOKEN` (صلاحية write)، وغيّر اسم المستخدم.

</div>

In [ ]:
HF_USER = "your-hf-username"
if HF_USER != "your-hf-username":
    from google.colab import userdata
    model.push_to_hub_gguf(f"{HF_USER}/masrufi-gguf", tokenizer, quantization_method = "q4_k_m",
                           token = userdata.get("HF_TOKEN"))
    print(f"على جهازك:  ollama run hf.co/{HF_USER}/masrufi-gguf:Q4_K_M")

<div dir="rtl" align="right">

## ٨. على جهازك

حط `masrufi.Q4_K_M.gguf` و `Modelfile` داخل مجلد مشروع **مصروفي**، وافتح الطرفية في نفس المجلد:

```
ollama create masrufi -f Modelfile
python eval.py qwen3:1.7b masrufi
python agent.py
```

الأمر الثاني يقارن النموذج الأصلي بنموذجك على جهازك، والثالث يشغّل الوكيل اللي يسجّل مصاريفك.

### لو النتيجة ما عجبتك
- **أضف أمثلة** للحالات اللي غلط فيها، وأعد التدريب. البيانات أهم من أي إعداد.
- جرّب `num_train_epochs = 3`، وانتبه: لو صار ممتاز في التدريب وسيئ في الاختبار فهو **حفظ** الأمثلة.
- جرّب نموذج أكبر (`Qwen3-4B`) لو عندك وقت.

</div>